# 年度恢复模型：实际动作与费用标签
连续2019–2022年度OOF参考，原实际恢复上限.6、alpha门和原特征，2bp共同次日重置优势。2020/21/22分别只用此前成熟标签，最少20条且双类；固定C=.3、阈值.825，最多3拟合。未知保留，不借后年数据。

In [1]:
from pathlib import Path
import os,json,hashlib,warnings
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
from src.workflow.minute_action_value import _source
from src.workflow.minute_pit_allocator import _dataset
from src.workflow.pit_constituent_auxiliary_training_allocation import _smooth_weights
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec,FEATURE_COLUMNS
from src.backtest.full_path_tiered import replay_full_path
from src.backtest.stage1_ibkr import native_tiered_stage1
from src.backtest.qlib_continuous_position import QlibContinuousPosition
from src.analysis.recovery_training_labels import recovery_training_labels
from src.data.recovery_first_hour import build_recovery_first_hour,FIELDS
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/annual_recovery_training_20261015';p=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013';os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
early,ea=_source(root,p['early_source'],['analysis/oof_predictions','models/2019_71']);later,la=_source(root,p['oof_source'],['analysis/oof_predictions']);market,ma=_source(root,p['price_source'],['inputs/frame','inputs/plan']);training,ta=_source(root,p['training_source'],['inputs/daily_frame']);action,aa=_source(root,p['action_source'],['inputs/plan','controller/source_recovery_selection'])
pred=pd.concat([early['analysis/oof_predictions'],later['analysis/oof_predictions']]).sort_index();assert len(pred)==1008 and not pred.index.duplicated().any();model=early['models/2019_71'];returns=training['inputs/daily_frame'].loc[pred.index,['soxl_forward_open_return','soxx_forward_open_return']];returns.columns=['soxl_return','soxx_return']
targets=_smooth_weights(pred.drop(columns=['trade_date','model_year']),spec=model.spec,candidate=model.candidate,asset_returns=returns);frame=market['inputs/frame'].set_index('signal_date').reindex(pred.index).rename_axis('signal_date').reset_index();assert pd.DatetimeIndex(frame.trade_date).equals(pd.DatetimeIndex(pred.trade_date))
for col in targets:frame[col]=targets[col].to_numpy()
assert market['inputs/plan']['tariff']==p['tariff'];recovery=IntradayRecoveryDirectAllocationSpec(root=root,**action['inputs/plan']['recovery_spec']);spec=Stage1OpenActionValueSpec(root=root,**action['inputs/plan']['action_spec']);selection=action['controller/source_recovery_selection'];assert selection['logistic_c']==.3 and selection['probability_threshold']==.825 and selection['action_beta']==.6
before_feature_repair=frame.copy()
feature_source=market['inputs/plan'];hour,hour_manifest=build_recovery_first_hour(source_manifest=root/feature_source['source_manifest'],source_sha256=feature_source['source_id'],provider_uri=root/'data/qlib/annual_recovery_first_hour_20261015',as_of=p['as_of'],workspace_root=root)
values=hour.reindex(pd.DatetimeIndex(frame.trade_date));assert values.first_hour_ready.notna().all()
for field in FIELDS:frame[field]=values[field].to_numpy()
frame['first_hour_ready']=values.first_hour_ready.to_numpy();frame['recovery_feature_available_at']=values.recovery_feature_available_at.to_numpy()
assert (pd.to_datetime(frame.recovery_feature_available_at)<=pd.to_datetime(frame.trade_date).dt.tz_localize('America/New_York')+pd.Timedelta(hours=10,minutes=31)).all()
resume=json.loads((card/'run_state.json').read_text()) if (card/'run_state.json').exists() else {}
controller={'frozen_open_action_value_model':None,'frozen_recovery_model':None}
if resume:
 with R.uri_context(str(card/'qlib_recorder')):
  old=R.get_recorder(recorder_id=resume['recorder_id'],experiment_id=resume['experiment_id']);assert old.load_object('inputs/plan')==p;book=old.load_object('analysis/native_reference');old_frame=old.load_object('inputs/frame')
  unchanged=[name for name in old_frame if not name.startswith('early_')];pd.testing.assert_frame_equal(frame[unchanged],old_frame[unchanged])
else:
 replay=replay_full_path(frame,models=None,plan=p,slippage_bps=2,controller=controller,recovery_spec=recovery,action_spec=spec,selection=selection);book=native_tiered_stage1(frame,replay,provider_uri=provider,tariff=p['tariff'],slippage_bps=2,initial_cash=p['initial_cash'],freq='1min',opening_offset=pd.Timedelta(hours=9,minutes=47),position_type=QlibContinuousPosition);book['targets']=replay['targets']
labels=recovery_training_labels(frame,book,plan=p,recovery_spec=recovery,action_spec=spec,selection=selection,slippage_bps=2)
x=labels.set_index('trade_date');ds=_dataset(x[list(FEATURE_COLUMNS)],x[['net_advantage']],pd.DataFrame(index=x.index),x[['eligible','label_available_at']],{str(year):('2019-01-01',f'{year-1}-12-31') for year in p['training_years']})
fitted={};audits=[]
kwargs={'experiment_name':card.name,'uri':str(card/'qlib_recorder')}
if resume:kwargs.update(recorder_id=resume['recorder_id'],resume=True)
with R.start(**kwargs):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,source_audits={'early':ea,'later':la,'market':ma,'training':ta,'action':aa},frame=frame,selection=selection,hour_manifest=hour_manifest,hour_features=hour,feature_builder=(root/'src/data/recovery_first_hour.py').read_text(),label_code=(root/'src/analysis/recovery_training_labels.py').read_text());rec.save_objects(artifact_path='analysis',native_reference=book,labels=labels,targets=targets)
 for year in p['training_years']:
  prepared=ds.prepare(str(year),col_set=['feature','label','meta']);cutoff=pd.Timestamp(f'{year}-01-01',tz='America/New_York');valid=prepared[('meta','eligible')].astype(bool)&pd.to_datetime(prepared[('meta','label_available_at')]).lt(cutoff)&np.isfinite(prepared[('label','net_advantage')])&np.isfinite(prepared['feature']).all(axis=1);train=prepared.loc[valid];y=train[('label','net_advantage')].gt(0)
  audit={'year':year,'rows':len(train),'positive_rows':int(y.sum()),'last_label_available_at':str(train[('meta','label_available_at')].max()),'cutoff':str(cutoff),'C':selection['logistic_c'],'threshold':selection['probability_threshold'],'minimum_rows':p['minimum_recovery_rows']}
  if len(train)<p['minimum_recovery_rows'] or y.nunique()<2:audit['status']='needs_data';fitted[year]=None
  else:
   model=make_pipeline(SimpleImputer(),StandardScaler(),LogisticRegression(C=selection['logistic_c'],class_weight='balanced',max_iter=2000,random_state=42))
   with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always',ConvergenceWarning);model.fit(train['feature'],y)
    if any(issubclass(w.category,ConvergenceWarning) for w in caught):raise ValueError('fixed model failed convergence')
   fitted[year]=model;audit.update(status='fitted',iterations=model[-1].n_iter_.tolist());rec.save_objects(artifact_path='models',**{str(year):model})
  audits.append(audit);rec.log_metrics(completed_fits=sum(a['status']=='fitted' for a in audits))
 rec.save_objects(artifact_path='analysis',training_audit=audits);rec.log_metrics(native_cases=1,registered_artifact_count=8+sum(a['status']=='fitted' for a in audits))
coverage=[]
for year,g in labels.groupby(labels.trade_date.dt.year):coverage.append({'year':year,'reference_days':len(g),'eligible':int(g.eligible.sum()),'known_labels':int(g.net_advantage.notna().sum()),'positive_labels':int(g.net_advantage.gt(0).sum())})
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','models/*','analysis/native_reference','analysis/labels','analysis/targets']};summary={'experiment':card.name,'decision':'annual_recovery_models_need_account_validation' if all(a['status']=='fitted' for a in audits) else 'partial_annual_recovery_models_needs_data','new_fits':sum(a['status']=='fitted' for a in audits),'new_native_cases':1,'reference_sessions':len(frame),'first_hour_qualified_sessions':int(frame.first_hour_ready.eq(1).sum()),'reference_reused_after_data_repair':bool(resume),'all_year_models_available':all(a['status']=='fitted' for a in audits),'provider_uri':str(provider),'region':'us','as_of':p['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'scope':p['scope'],'objective_achieved':False}
pd.DataFrame(audits).to_csv(card/'training_audit.csv',index=False);pd.DataFrame(coverage).to_csv(card/'label_coverage.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}));binding=json.loads((root/p['early_source']['card']/'nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2));a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2));summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(pd.DataFrame(audits).to_string(index=False));print(pd.DataFrame(coverage).to_string(index=False))


# 年度恢复训练：修复首小时接入，完成两年模型

本轮完成1个连续1008日Qlib原生标签参考账户，2次恢复分类器拟合。参考覆盖2019–2022，年度OOF配置按交易年接续，目标平滑及持仓/现金/风险状态不跨年重置，10万美元、固定IBKR Pro Tiered2bp。参考不含学习型开盘/恢复控制器，避免由未来控制器产生训练状态。

首次接入发现：父full_path价格表为日频模型构造，early_*只挂接外层2023+；2019–2022的首小时字段全空。这是本次调用的输入覆盖不匹配，不是原始分钟行情不存在；当时未拟合任何模型，也没有把空标签填0。已从同一冻结NAS分钟来源重建前60根0931至1030右边界bar，经公开仿射价和拆股调整量生成原定义字段，再写入Qlib并原生读回。价格字段和无恢复参考的状态/现金逐日保持一致，原生参考账户没有重跑。

1008个参考日中1007日首小时合格；已知SOXX2020-03-16异常使配对首小时字段为未知，不插值、不删压力日。字段可用时间固定为1031，标签恢复成交在1032。构建器测试确认1030之后bar不影响首小时，异常bar使整组保留NaN。历史bar加一分钟可用假设与公开调整价量不是真实首收或历史复权版本PIT认证。

动作标签固定对应实际部署参数：恢复beta最多0.6、原alpha门0.25，满足原回撤/风险上限资格才构造；不再用较高teacher beta代表较低实际动作。开盘已完成交易后，以同一股数、现金及已扣开盘费用状态分出恢复与HOLD两支，均计入1032执行及次日0947共同目标重置费用；标签到次日0949才成熟。该局部重置优势不是完整自主策略收益。

| 年度模型 | 截止前成熟样本 | 正类 | 最后标签成熟ET | 状态 |
|---|---:|---:|---|---|
| 2020 | 19 | 9 | 2019-08-16 09:49 | needs_data，未拟合 |
| 2021 | 174 | 102 | 2020-12-23 09:49 | 已拟合 |
| 2022 | 201 | 116 | 2021-12-31 09:49 | 已拟合 |

固定C=.3、balanced、random_state=42、max_iter=2000及原概率阈值.825；2021/2022分别18/15次迭代收敛。沿用原至少20条且双类门槛，2020没有因为只差1条就降门槛，没有向2021借标签。所用参数来自既有历史研究选择，年度训练时序合格不等于独立策略验证。

决策partial_annual_recovery_models_needs_data。仅2021/2022可进入后续完整账户诊断；2020仍无合格恢复模型，不能把缺失默认为成功策略。下一项需固定同输入对照、0/2/5bp各自结算，明确参考2019预热和评价期起点，不把新预热效果归给模型；不调整阈值或重新筛样本。现阶段没有新的策略收益证据，也不能称完整历史控制器已重建。

2项既有动作费用测试及1项新增首小时合同测试通过。模型、标签、源数据接口修复与原生参考证据均在Qlib Recorder，NAS冷恢复逐对象哈希通过。生产未改，稳定净SOXX超额目标未完成。
